# 🏆 iRage AlgoArena 2026 — Solution Walkthrough

**Author:** Gunjan Moradiya · Final Year B.Tech CS (Data Science), DJSCE Mumbai  
**Result:** Top 10 out of 500+ participants

---

## Competition Context

The task was to predict **short-horizon financial returns** from hundreds of high-dimensional signals with temporal structure.

The dataset contained:
- `ID` — unique row identifier
- `TARGET` — short-horizon return (our prediction target)
- `CV_GROUP` — temporal group labels for cross-validation
- `S01_LagT1`, `S01_LagT2`, `S02_LagT1`, `S03_LagT3`, … — signal features across multiple signals and lag windows

This notebook walks through every step of the solution with detailed explanations of the *why* behind each decision.

## Step 0: Setup and Data Loading

We load the competition datasets. **Note:** The actual competition data is proprietary and not included in this repository. Below we show the expected schema so you can adapt this to your own dataset.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

# ============================================================
# DATA LOADING
# The competition data is proprietary and not included here.
# Expected schema:
#   train.csv: ID, TARGET, CV_GROUP, S01_LagT1, S01_LagT2, S01_LagT3, S02_LagT1, ...
#   test.csv:  ID, S01_LagT1, S01_LagT2, S01_LagT3, S02_LagT1, ...
#
# Replace the paths below with your own data paths.
# ============================================================

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'\nColumns (first 20): {list(train.columns[:20])}')
print(f'\nCV_GROUP unique values: {train["CV_GROUP"].nunique()}')

## Step 1: Feature Selection — Correlation-Based Filtering

**What:** Compute `|pearson_corr(feature, TARGET)|` for every raw feature, sort descending, keep the top 120.

**Why this matters:**
The raw feature space contains hundreds of signal columns, but most are noise. In financial return prediction, noisy features don't just waste computation — they actively degrade gradient-boosted tree models by:
1. Diluting split quality (the algorithm wastes splits on uninformative columns)
2. Increasing variance without reducing bias

A hard filter on univariate correlation is computationally cheap and effective at retaining features most likely to carry genuine signal. We chose 120 as a threshold that balances coverage (keeping enough features for downstream engineering) against noise reduction.

**Why Pearson and not mutual information or other measures?**
For tree-based models, monotonic relationships (captured by Pearson) are the primary signal source. Pearson correlation is also orders of magnitude faster to compute than MI, which matters when iterating quickly in a competition.

In [ ]:
# Identify feature columns (exclude metadata columns)
exclude_cols = {'ID', 'TARGET', 'CV_GROUP'}
feature_cols = [c for c in train.columns if c not in exclude_cols]

# Compute absolute Pearson correlation with TARGET
correlations = train[feature_cols].corrwith(train['TARGET']).abs()
correlations = correlations.sort_values(ascending=False)

# Select top 120 features
top_features = correlations.head(120).index.tolist()

print(f'Total raw features: {len(feature_cols)}')
print(f'Selected top features: {len(top_features)}')
print(f'\nTop 10 features by |correlation|:')
print(correlations.head(10))

## Step 2: Momentum & Acceleration Features

**What:** For every `LagT1` feature in the top 120, if `LagT2` and `LagT3` also exist:
- **Momentum** = `LagT1 - LagT2` (first discrete difference → rate of change)
- **Acceleration** = `LagT1 - 2·LagT2 + LagT3` (second discrete difference → change in momentum)

**Why this matters:**
Raw lag values tell you *where a signal currently is*. But in financial markets, the *derivative* often carries more predictive information than the level:

- A signal at value 5.0 that was 3.0 yesterday (momentum = +2.0) carries fundamentally different information than one at 5.0 that was 7.0 yesterday (momentum = -2.0)
- Acceleration captures whether a trend is *strengthening* or *fading* — a decelerating uptrend (positive momentum, negative acceleration) may signal an impending reversal

This mirrors how quantitative traders interpret price action: not just the price, but the rate and acceleration of price change.

**Note on the discrete second derivative formula:**
`f''(t) ≈ f(t) - 2f(t-1) + f(t-2)` — this is the standard central difference approximation for the second derivative.

In [ ]:
def add_momentum_features(df, top_features):
    """Add momentum (1st diff) and acceleration (2nd diff) features."""
    new_features = {}
    
    # Find LagT1 features among our selected features
    lag1_features = [f for f in top_features if '_LagT1' in f]
    
    for feat in lag1_features:
        base = feat.replace('_LagT1', '')
        lag2 = f'{base}_LagT2'
        lag3 = f'{base}_LagT3'
        
        # First difference: rate of change
        if lag2 in df.columns:
            momentum_name = f'momentum_{base}'
            new_features[momentum_name] = df[feat].values - df[lag2].values
        
        # Second difference: change in momentum (discrete second derivative)
        if lag2 in df.columns and lag3 in df.columns:
            accel_name = f'accel_{base}'
            new_features[accel_name] = (
                df[feat].values - 2 * df[lag2].values + df[lag3].values
            )
    
    return new_features


# Apply to both train and test
momentum_train = add_momentum_features(train, top_features)
momentum_test = add_momentum_features(test, top_features)

for name, values in momentum_train.items():
    train[name] = values
    test[name] = momentum_test[name]

momentum_feature_names = list(momentum_train.keys())
print(f'Created {len(momentum_feature_names)} momentum/acceleration features')
print(f'Examples: {momentum_feature_names[:6]}')

## Step 3: Intra-Signal Interaction Features

**What:** Take the top 20 features (by correlation rank). Compute all pairwise products for i < j → C(20, 2) = **190 interaction features**.

**Why this matters:**
Gradient-boosted trees find *axis-aligned* decision boundaries (splits on single features). But in financial data, the true predictive boundary often lies along a *diagonal* in feature space — i.e., a feature is only predictive *conditional on* another feature's value.

Pairwise products explicitly encode these multiplicative relationships:
- If `f1 > 0` and `f2 > 0`, their product is positive
- If they have opposite signs, the product is negative
- This lets the model capture conditional dependencies with fewer tree splits

**Why top 20 (not all 120)?**
C(120, 2) = 7,140 features — too many, and most would be noise × noise. Using only the top 20 ensures we're interacting the strongest signals, keeping the SNR high.

In [ ]:
# Take the top 20 features for intra-signal interactions
top_20 = top_features[:20]
intra_features = {}

for i in range(len(top_20)):
    for j in range(i + 1, len(top_20)):
        f1, f2 = top_20[i], top_20[j]
        name = f'intra_{f1}_x_{f2}'
        
        # Pairwise product captures multiplicative relationships
        train[name] = train[f1].values * train[f2].values
        test[name] = test[f1].values * test[f2].values
        intra_features[name] = True

intra_feature_names = list(intra_features.keys())
print(f'Created {len(intra_feature_names)} intra-signal interaction features')
print(f'(Expected: C(20,2) = {20*19//2})')

## Step 4: Cross-Signal Interaction Features

**What:**
- Take top 5 `S01` features × top 5 `S02` features → 25 cross-products (`cross12_`)
- Take top 5 `S01` features × top 5 `S03` features → 25 cross-products (`cross13_`)
- Total: **50 cross-signal features**

**Why this matters:**
Different signal families (`S01`, `S02`, `S03`) likely encode different aspects of market microstructure — for instance, one might capture order flow imbalance, another might capture volatility dynamics, and a third might reflect spread behavior.

Cross-signal products allow the model to detect **joint regimes** — situations where combinations of signals from different families are predictive in ways that no single signal family can express alone. For example, a high-momentum S01 signal combined with low-volatility S02 signal might be predictive of mean reversion, while the same S01 momentum combined with high S02 volatility might signal a breakout.

**Why only 5 per group?**
This keeps the feature count controlled (5×5×2 = 50) while focusing on the strongest representatives from each signal family.

In [ ]:
# Extract top features per signal family (preserving correlation-sorted order)
s01_feats = [f for f in top_features if f.startswith('S01_')][:5]
s02_feats = [f for f in top_features if f.startswith('S02_')][:5]
s03_feats = [f for f in top_features if f.startswith('S03_')][:5]

cross_feature_names = []

# S01 × S02 interactions
for f1 in s01_feats:
    for f2 in s02_feats:
        name = f'cross12_{f1}_x_{f2}'
        train[name] = train[f1].values * train[f2].values
        test[name] = test[f1].values * test[f2].values
        cross_feature_names.append(name)

# S01 × S03 interactions
for f1 in s01_feats:
    for f2 in s03_feats:
        name = f'cross13_{f1}_x_{f2}'
        train[name] = train[f1].values * train[f2].values
        test[name] = test[f1].values * test[f2].values
        cross_feature_names.append(name)

print(f'Created {len(cross_feature_names)} cross-signal interaction features')
print(f'  S01×S02: {len(s01_feats) * len(s02_feats)}')
print(f'  S01×S03: {len(s01_feats) * len(s03_feats)}')

## Compile Final Feature List

Combine all feature categories into a single list for modelling.

In [ ]:
# Combine all engineered feature names
all_features = (
    top_features              # 120 raw selected features
    + momentum_feature_names  # momentum & acceleration
    + intra_feature_names     # intra-signal interactions
    + cross_feature_names     # cross-signal interactions
)

print(f'Total features for modelling: {len(all_features)}')
print(f'  Raw selected:       {len(top_features)}')
print(f'  Momentum/Accel:     {len(momentum_feature_names)}')
print(f'  Intra interactions: {len(intra_feature_names)}')
print(f'  Cross interactions: {len(cross_feature_names)}')

## Step 5: Model A — Conservative (High Regularization)

**Setup:**
- LightGBM with `reg_alpha=5, reg_lambda=5` (strong L1 + L2 penalties)
- 5-fold GroupKFold on `CV_GROUP`, early stopping at 200 rounds

**Why GroupKFold?**
The `CV_GROUP` column encodes temporal structure. Standard KFold would randomly assign rows to folds, mixing time periods. In financial data, consecutive time periods are autocorrelated — if the model trains on data from time period T and validates on nearby data from T±1, it sees information that would not be available in real deployment. This is **temporal leakage**, and it inflates CV scores dramatically.

GroupKFold ensures that all rows from the same `CV_GROUP` land entirely in either training or validation, never both. This simulates the realistic scenario of training on past data and predicting unseen future periods.

**Why high regularization?**
Strong L1/L2 penalties produce smoother, more conservative predictions. The model won't overfit to noise in the training set, which is critical for financial data where the signal-to-noise ratio is inherently low. This model acts as the **stable backbone** of our ensemble.

In [ ]:
# ============================================================
# MODEL A: Conservative — high regularization (α=5, λ=5)
# Produces smoother predictions with lower variance
# ============================================================

params_A = {
    'n_estimators': 2000,
    'learning_rate': 0.02,
    'num_leaves': 128,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 5,        # Strong L1 regularization
    'reg_lambda': 5,       # Strong L2 regularization
    'random_state': 42,
    'verbose': -1,
}

gkf = GroupKFold(n_splits=5)
groups = train['CV_GROUP'].values

oof_A = np.zeros(len(train))
test_A = np.zeros(len(test))

X_train_all = train[all_features].values
y_train_all = train['TARGET'].values
X_test_all = test[all_features].values

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_all, y_train_all, groups)):
    print(f'\n{"="*60}')
    print(f'Model A — Fold {fold+1}/5')
    print(f'  Train: {len(train_idx):,} rows  |  Val: {len(val_idx):,} rows')
    print(f'{"="*60}')
    
    X_tr, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_tr, y_val = y_train_all[train_idx], y_train_all[val_idx]
    
    model_A = lgb.LGBMRegressor(**params_A)
    model_A.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='l2',
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=500),
        ],
    )
    
    oof_A[val_idx] = model_A.predict(X_val)
    test_A += model_A.predict(X_test_all)

# Average test predictions across 5 folds
test_A /= 5

mse_A = np.mean((oof_A - y_train_all) ** 2)
print(f'\nModel A OOF MSE: {mse_A:.6f}')

## Step 6: Model B — Aggressive (Low Regularization)

**What:** Identical to Model A in every way except regularization: `reg_alpha=0.5, reg_lambda=0.5`.

**Why a second model with lower regularization?**
Model A intentionally smooths its predictions, sacrificing some ability to capture fine-grained patterns. Model B, with weaker penalties, fits the data more aggressively. It will:
- Capture patterns that Model A's regularization smoothed away
- Have higher variance (more prone to overfitting)
- Make *different errors* than Model A

The key insight for the ensemble is that **the models err differently**. High-reg models underfit subtleties; low-reg models overfit noise. By varying *only* the regularization strength, we create two models that are architecturally identical but have decorrelated error distributions — the ideal setup for a blending ensemble.

In [ ]:
# ============================================================
# MODEL B: Aggressive — low regularization (α=0.5, λ=0.5)
# Captures finer-grained patterns at the cost of higher variance
# ============================================================

params_B = {
    'n_estimators': 2000,
    'learning_rate': 0.02,
    'num_leaves': 128,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,      # Weak L1 regularization
    'reg_lambda': 0.5,     # Weak L2 regularization
    'random_state': 42,
    'verbose': -1,
}

oof_B = np.zeros(len(train))
test_B = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_all, y_train_all, groups)):
    print(f'\n{"="*60}')
    print(f'Model B — Fold {fold+1}/5')
    print(f'  Train: {len(train_idx):,} rows  |  Val: {len(val_idx):,} rows')
    print(f'{"="*60}')
    
    X_tr, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_tr, y_val = y_train_all[train_idx], y_train_all[val_idx]
    
    model_B = lgb.LGBMRegressor(**params_B)
    model_B.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='l2',
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=500),
        ],
    )
    
    oof_B[val_idx] = model_B.predict(X_val)
    test_B += model_B.predict(X_test_all)

# Average test predictions across 5 folds
test_B /= 5

mse_B = np.mean((oof_B - y_train_all) ** 2)
print(f'\nModel B OOF MSE: {mse_B:.6f}')

## Step 7: Final Ensemble — Weighted Blend

**Formula:** `pred_final = 0.72 × pred_A + 0.28 × pred_B`

**Why 72:28 and not 50:50?**

This weighting reflects a deliberate **bias-variance tradeoff** decision:

- **Financial return prediction penalizes large errors disproportionately.** Overfit predictions don't just lose accuracy — they can produce catastrophic outliers that tank overall performance.
- **Model A (conservative)** produces stable predictions with lower variance. It should carry the majority of the weight because its predictions are more *reliable*.
- **Model B (aggressive)** adds incremental signal from patterns that Model A's regularization smoothed away. The 28% allocation lets it contribute where it sees something genuine, without letting its higher variance dominate the blend.

This is consistent with a general principle in noisy prediction tasks: **trust the stable model by default, but let the expressive model contribute at the margin.** The optimal weight was determined by evaluating different blend ratios on the OOF predictions.

In [ ]:
# ============================================================
# ENSEMBLE: 72% Conservative + 28% Aggressive
# Stability-weighted blend for noisy financial predictions
# ============================================================

weight_A = 0.72
weight_B = 0.28

# Final test predictions
pred_final = weight_A * test_A + weight_B * test_B

# Also compute blended OOF for validation
oof_final = weight_A * oof_A + weight_B * oof_B
mse_ensemble = np.mean((oof_final - y_train_all) ** 2)

print(f'Model A OOF MSE:    {mse_A:.6f}')
print(f'Model B OOF MSE:    {mse_B:.6f}')
print(f'Ensemble OOF MSE:   {mse_ensemble:.6f}')
print(f'\nEnsemble weight: {weight_A:.0%} A + {weight_B:.0%} B')

## Step 8: Generate Submission

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'ID': test['ID'],
    'TARGET': pred_final,
})

submission.to_csv('submission.csv', index=False)
print(f'Submission shape: {submission.shape}')
print(f'\nPrediction statistics:')
print(submission['TARGET'].describe())

---

## Results & Key Takeaways

### Final Standing: Top 10 / 500+ participants

### What worked:

1. **Respecting temporal structure with GroupKFold.** This was the single most important decision. CV scores aligned closely with leaderboard scores, confirming we were measuring genuine out-of-sample performance, not leakage.

2. **Hypothesis-driven feature engineering.** Every engineered feature had a financial rationale:
   - Momentum/acceleration capture signal *dynamics*, not just levels
   - Interaction features capture *conditional* relationships between signals
   - Limiting interaction counts preserved the signal-to-noise ratio

3. **The dual-regularization ensemble.** By varying only regularization, we created two models with decorrelated errors — the ideal setup for blending. The 72:28 weighting correctly prioritized stability over expressiveness.

4. **Aggressive noise filtering.** Starting with 120 features (not all of them) meant every downstream operation — momentum, interactions — operated on cleaner inputs. The cascading effect of early noise removal was significant.

### What I'd explore with more time:
- Rolling statistics (mean, std) over the lag dimension per signal
- Rank-transformed features to handle outliers in financial data
- Broader hyperparameter search for ensemble weights
- Additional model diversity (e.g., CatBoost or neural network as a third model)